In [ ]:
from pathlib import Path
import sys
import json
from datetime import datetime
from typing import Dict
project_root = Path.cwd().resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from Llm.llm_loader import LLM
import pandas as pd
from dotenv import load_dotenv
from tqdm import tqdm
import random
GLOBAL_RANDOM_SEED = 20260324
OLLAMA_NUM_CTX = 65536
load_dotenv(project_root / '.env')

In [ ]:
def load_data(project_root, dataset_name):
    datasets = [
        (
            'two_table',
            project_root / 'Data' / dataset_name / 'Synthesized_two_table_with_spider_db_id.json',
            pd.read_json(project_root / 'Data' / dataset_name / 'Synthesized_two_table_with_spider_db_id.json')
        ),
        (
            'three_table',
            project_root / 'Data' / dataset_name / 'Synthesized_three_table_with_spider_db_id.json',
            pd.read_json(project_root / 'Data' / dataset_name / 'Synthesized_three_table_with_spider_db_id.json')
        ),
    ]
    return datasets

def load_database_schema(path: Path) -> Dict: 
    with open(path, "r") as f:
        database_schemas = json.load(f)
    return database_schemas

In [ ]:
def append_log_entry(log_records, dataset_name, data_split, row, response_text,Answer_llm,log_path, prompt_chars, schema_chars, sample_seed):
    log_records.append(
        {
            'model': Answer_llm.model_name,
            'provider': Answer_llm.provider,
            'id': f"{dataset_name}-{data_split}-{row['id_']}",
            'spider_db_id': row['Spider_db_id'],
            'question': row['Question'],
            'prompt_chars': prompt_chars,
            'schema_chars': schema_chars,
            'sample_seed': sample_seed,
            'ollama_num_ctx': getattr(Answer_llm, 'num_ctx', None),
            'response': response_text,
        }
    )
    log_path.write_text(json.dumps(log_records, ensure_ascii=False, indent=2), encoding='utf-8')

def database_schema_to_string(database_schemas: Dict, num: int, target_id: str, rng: random.Random) -> str:
    all_keys = list(database_schemas.keys())
    if num > len(database_schemas.keys()):
        candidate_keys = all_keys
    else:
        candidate_keys = all_keys.copy()
        candidate_keys.remove(target_id)
        candidate_keys = rng.sample(candidate_keys, num)
        candidate_keys.append(target_id)
    rng.shuffle(candidate_keys)
    schemas_info_string = ""
    for key in candidate_keys:
        s_string = database_schemas[key]
        schemas_info_string = schemas_info_string + s_string + "\n"+ "="*80 + "\n"
    return schemas_info_string.strip()

In [ ]:
def run_(prompt_path, log_path, datasets,dataset_name,database_schema_path,num_noise,Answer_llm):
    prompt_template = prompt_path.read_text(encoding='utf-8').strip()
    log_records = []
    log_path.write_text(json.dumps(log_records, ensure_ascii=False, indent=2), encoding='utf-8')
    database_schemas = load_database_schema(database_schema_path)
    for data_split, data_source, df in datasets:
        for _, row in tqdm(df.iterrows(), total=len(df), desc=data_split):
            sample_seed = GLOBAL_RANDOM_SEED + int(row['id_'])
            rng = random.Random(sample_seed)
            schemas_string = database_schema_to_string(database_schemas, num_noise, row['Spider_db_id'], rng)
            prompt = (
                prompt_template
                .replace('{DATABASE_SCHEMAS}', schemas_string)
                .replace('{QUESTION}', row['Question'])
                .replace('{HINT}', 'No hint')
            )
            prompt_chars = len(prompt)
            schema_chars = len(schemas_string)
            res = Answer_llm.query(prompt)
            append_log_entry(log_records, dataset_name, data_split, row, res, Answer_llm, log_path, prompt_chars, schema_chars, sample_seed)

    print(f'All responses have been saved to {log_path}')

In [ ]:
dataset_name = 'MMQA'
datasets = load_data(project_root, dataset_name)

In [ ]:
ollama_llm_lists = ['llama3.2:3b']
gpt_llm_lists = ['gpt-4.1-nano']

In [ ]:
method_ = 'zero_shot'
num_noise = 999
database_schema_path = project_root / 'Data' / 'MMQA' / 'Database_schemas_summary.json'
prompt_path = project_root / 'Templates' / method_ / 'find_relevant_database_baseline.txt'

In [ ]:
for model_name in ollama_llm_lists:
    Answer_llm = LLM(model_name = model_name, provider= 'ollama', num_ctx=OLLAMA_NUM_CTX)
    run_id = datetime.now().strftime('%Y%m%d_%H%M%S')
    logs_dir = project_root / 'Logs' / model_name / 'Database_Retrival'
    logs_dir.mkdir(parents=True,exist_ok=True)
    log_path = logs_dir / f'{method_}_baseline_{num_noise}_{dataset_name}_{run_id}.json'
    run_(prompt_path, log_path, datasets,dataset_name,database_schema_path,num_noise,Answer_llm)

In [ ]:
for model_name in gpt_llm_lists:
    Answer_llm = LLM(model_name = model_name, provider= 'openai')
    run_id = datetime.now().strftime('%Y%m%d_%H%M%S')
    logs_dir = project_root / 'Logs' / model_name / 'Database_Retrival'
    logs_dir.mkdir(parents=True,exist_ok=True)
    log_path = logs_dir / f'{method_}_baseline_{num_noise}_{dataset_name}_{run_id}.json'
    run_(prompt_path, log_path, datasets,dataset_name,database_schema_path,num_noise,Answer_llm)